### P1 — Regression

In [1]:
from mibici.modeling.p1 import load_p1_data

daily_demand = load_p1_data()

analysis_data = daily_demand.copy()
analysis_data["day_of_week"] = analysis_data["date"].dt.dayofweek

2026-10-17 23:58:33.445 | INFO     | mibici.config:<module>:11 - PROJ_ROOT path is: /home/jovyan/notebooks/mibici/mibici_predictions


In [2]:
station_weekday_profile = analysis_data.pivot_table(
    index="station_id",
    columns="day_of_week",
    values="departures",
    aggfunc="mean",
).reindex(columns=range(7))

station_weekday_profile.columns = [
    "Monday", "Tuesday", "Wednesday", "Thursday",
    "Friday", "Saturday", "Sunday",
]

display(station_weekday_profile.round(2).head(10))

,Monday,Tuesday,Wednesday,Thursday,Friday,Saturday,Sunday
station_id,,,,,,,
2,54.17,58.28,52.44,55.12,64.24,45.41,26.35
3,47.56,56.33,52.00,49.18,52.59,37.65,26.53
4,64.00,69.83,62.89,64.82,67.18,44.12,57.12
5,54.44,64.06,57.06,59.71,59.06,37.24,29.29
6,36.33,41.11,36.17,38.00,36.35,28.71,25.35
8,29.78,32.83,29.11,29.24,31.00,23.59,16.06
9,43.78,45.50,45.50,42.53,46.29,33.88,26.82
10,39.89,46.00,44.17,43.41,44.65,32.29,19.82
11,192.78,201.67,199.50,199.12,190.00,138.35,109.06


En las estaciones mostradas, el promedio de salidas cambia según
el día de la semana. La magnitud y el patrón difieren por estación.
Se propone evaluar estación y día de la semana como predictores.

In [3]:
analysis_data = analysis_data.sort_values(
    ["station_id", "date"]
).copy()

analysis_data["departures_lag_1"] = (
    analysis_data.groupby("station_id")["departures"].shift(1)
)

display(
    analysis_data[
        ["station_id", "date", "departures", "departures_lag_1"]
    ].head(10)
)

,station_id,date,departures,departures_lag_1
0,2,2026-06-01,67,NaN
1,2,2026-06-02,46,67.0
2,2,2026-06-03,58,46.0
3,2,2026-06-04,47,58.0
4,2,2026-06-05,40,47.0
5,2,2026-06-06,50,40.0
6,2,2026-06-07,31,50.0
7,2,2026-06-08,47,31.0
8,2,2026-06-09,64,47.0
9,2,2026-06-10,52,64.0


In [4]:
lag_pairs = analysis_data[
    ["departures", "departures_lag_1"]
].dropna()

display(lag_pairs.corr())

,departures,departures_lag_1
departures,1.000000,0.894876
departures_lag_1,0.894876,1.000000


In [5]:
station_lag_correlation = (
    analysis_data.groupby("station_id")[
        ["departures", "departures_lag_1"]
    ]
    .corr()
    .loc[(slice(None), "departures"), "departures_lag_1"]
    .droplevel(1)
    .rename("lag_1_correlation")
)

display(station_lag_correlation.describe())

count    470.000000
mean       0.318590
std        0.174030
min       -0.212695
25%        0.211280
50%        0.307110
75%        0.432426
max        0.905644
Name: lag_1_correlation, dtype: float64

El rezago de un día presenta correlación global alta (0.895),
pero una mediana por estación de 0.307. Se conserva como variable
candidata; su utilidad se evaluará mediante validación temporal.

In [6]:
analysis_data["departures_lag_7"] = (
    analysis_data.groupby("station_id")["departures"].shift(7)
)

In [7]:
comparison_data = analysis_data.dropna(
    subset=["departures", "departures_lag_1", "departures_lag_7"]
)

display(
    comparison_data[
        ["departures", "departures_lag_1", "departures_lag_7"]
    ].corr()
)

,departures,departures_lag_1,departures_lag_7
departures,1.000000,0.893690,0.912045
departures_lag_1,0.893690,1.000000,0.872238
departures_lag_7,0.912045,0.872238,1.000000


In [8]:
station_correlations = (
    comparison_data.groupby("station_id")[
        ["departures", "departures_lag_1", "departures_lag_7"]
    ]
    .corr()
    .xs("departures", level=1)[
        ["departures_lag_1", "departures_lag_7"]
    ]
)

display(station_correlations.describe())

,departures_lag_1,departures_lag_7
count,470.000000,470.000000
mean,0.303024,0.435742
std,0.175140,0.189276
min,-0.216708,-0.103627
25%,0.199354,0.310428
50%,0.304906,0.440137
75%,0.415428,0.586191
max,0.904482,0.879262


In [9]:
analysis_data["departures_mean_7"] = (
    analysis_data.groupby("station_id")["departures"]
    .transform(
        lambda values: values.shift(1).rolling(
            window=7,
            min_periods=7,
        ).mean()
    )
)

In [10]:
display(
    analysis_data[
        [
            "station_id", "date", "departures",
            "departures_lag_1", "departures_lag_7",
            "departures_mean_7",
        ]
    ].head(12)
)

,station_id,date,departures,departures_lag_1,departures_lag_7,departures_mean_7
0,2,2026-06-01,67,NaN,NaN,NaN
1,2,2026-06-02,46,67.0,NaN,NaN
2,2,2026-06-03,58,46.0,NaN,NaN
3,2,2026-06-04,47,58.0,NaN,NaN
4,2,2026-06-05,40,47.0,NaN,NaN
5,2,2026-06-06,50,40.0,NaN,NaN
6,2,2026-06-07,31,50.0,NaN,NaN
7,2,2026-06-08,47,31.0,67.0,48.428571
8,2,2026-06-09,64,47.0,46.0,45.571429
9,2,2026-06-10,52,64.0,58.0,48.142857


In [11]:
feature_columns = [
    "departures_lag_1",
    "departures_lag_7",
    "departures_mean_7",
]

comparison_data = analysis_data.dropna(
    subset=["departures"] + feature_columns
)

station_correlations = (
    comparison_data.groupby("station_id")[
        ["departures"] + feature_columns
    ]
    .corr()
    .xs("departures", level=1)[feature_columns]
)

display(station_correlations.describe())

,departures_lag_1,departures_lag_7,departures_mean_7
count,470.000000,470.000000,470.000000
mean,0.303024,0.435742,0.353537
std,0.175140,0.189276,0.184503
min,-0.216708,-0.103627,-0.326032
25%,0.199354,0.310428,0.237631
50%,0.304906,0.440137,0.354798
75%,0.415428,0.586191,0.483448
max,0.904482,0.879262,0.875894


#### Conclusiones para P1

Se proponen como variables candidatas: estación, día de la semana,
salidas del día anterior, salidas de hace siete días y promedio
de los siete días anteriores.

El rezago semanal presenta la mayor correlación mediana por estación.
Se utilizará también como referencia de predicción.

Los faltantes iniciales de los rezagos representan historia
no disponible y no se rellenarán automáticamente con cero.

La utilidad de las variables se comprobará mediante validación
temporal; la correlación no determina su selección final.

### P2 — Classification